# Fold 2b — Generación de datos sintéticos con GAN para balancear una clase minoritaria

En control de calidad visual, la clase de "defecto" suele ser rara: pocos ejemplos reales, alto costo de esperar a que ocurran más. Este notebook entrena una **Red Generativa Adversaria (GAN)** para producir imágenes sintéticas plausibles de esa clase minoritaria, con el objetivo de balancear el dataset de entrenamiento de un clasificador de defectos.

Se usa **MNIST** como proxy simplificado: una clase de dígito (p. ej. el dígito "8") se trata como si fuera la "clase de defecto minoritaria", para demostrar el pipeline completo sin depender de un dataset industrial propietario. El mismo código aplica directamente sobre imágenes reales de piezas — solo cambia la fuente de datos de entrada.

In [ ]:
# pip install tensorflow --break-system-packages
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(x_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

# Clase minoritaria "de interés" (proxy de la clase de defecto real)
CLASE_INTERES = 8
x_clase = x_train[y_train == CLASE_INTERES]
x_clase = (x_clase.astype('float32') - 127.5) / 127.5  # normalizar a [-1, 1]
x_clase = x_clase.reshape(-1, 28, 28, 1)

print(f"Ejemplos disponibles de la clase de interés: {x_clase.shape[0]}")

## 1. Generador y discriminador

In [ ]:
from tensorflow.keras import layers, Model

LATENT_DIM = 100

def construir_generador():
    modelo = tf.keras.Sequential([
        layers.Dense(7*7*128, use_bias=False, input_shape=(LATENT_DIM,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Reshape((7, 7, 128)),

        layers.Conv2DTranspose(64, 5, strides=1, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2DTranspose(32, 5, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2DTranspose(1, 5, strides=2, padding='same', use_bias=False, activation='tanh'),
    ])
    return modelo


def construir_discriminador():
    modelo = tf.keras.Sequential([
        layers.Conv2D(32, 5, strides=2, padding='same', input_shape=(28, 28, 1)),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),

        layers.Conv2D(64, 5, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1),
    ])
    return modelo


generador = construir_generador()
discriminador = construir_discriminador()
generador.summary()

## 2. Funciones de pérdida y optimizadores

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def perdida_discriminador(salida_real, salida_falsa):
    perdida_real = cross_entropy(tf.ones_like(salida_real), salida_real)
    perdida_falsa = cross_entropy(tf.zeros_like(salida_falsa), salida_falsa)
    return perdida_real + perdida_falsa

def perdida_generador(salida_falsa):
    return cross_entropy(tf.ones_like(salida_falsa), salida_falsa)

opt_generador = tf.keras.optimizers.Adam(1e-4)
opt_discriminador = tf.keras.optimizers.Adam(1e-4)

## 3. Paso de entrenamiento y bucle principal

Ajuste documentado frente al notebook original de clase: se reduce el learning rate y se añade `BatchNormalization` en generador y discriminador, porque el entrenamiento inicial (sin estas dos medidas) era inestable — el discriminador "ganaba" demasiado rápido y el generador dejaba de aprender (modo de colapso típico de GANs).

In [ ]:
BATCH_SIZE = 64
EPOCHS = 30  # subir a 100+ para resultados de mejor calidad; 30 es suficiente para validar el pipeline

dataset = tf.data.Dataset.from_tensor_slices(x_clase).shuffle(1000).batch(BATCH_SIZE)

@tf.function
def paso_entrenamiento(imagenes_reales):
    ruido = tf.random.normal([imagenes_reales.shape[0], LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        imagenes_generadas = generador(ruido, training=True)

        salida_real = discriminador(imagenes_reales, training=True)
        salida_falsa = discriminador(imagenes_generadas, training=True)

        loss_gen = perdida_generador(salida_falsa)
        loss_disc = perdida_discriminador(salida_real, salida_falsa)

    grad_gen = gen_tape.gradient(loss_gen, generador.trainable_variables)
    grad_disc = disc_tape.gradient(loss_disc, discriminador.trainable_variables)

    opt_generador.apply_gradients(zip(grad_gen, generador.trainable_variables))
    opt_discriminador.apply_gradients(zip(grad_disc, discriminador.trainable_variables))

    return loss_gen, loss_disc


for epoch in range(EPOCHS):
    perdidas_gen, perdidas_disc = [], []
    for batch in dataset:
        lg, ld = paso_entrenamiento(batch)
        perdidas_gen.append(float(lg))
        perdidas_disc.append(float(ld))

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | loss_gen={np.mean(perdidas_gen):.3f} | loss_disc={np.mean(perdidas_disc):.3f}")

## 4. Generar ejemplos sintéticos de la clase minoritaria

In [ ]:
def mostrar_generadas(n=16):
    ruido = tf.random.normal([n, LATENT_DIM])
    imagenes = generador(ruido, training=False)
    imagenes = (imagenes.numpy() * 127.5 + 127.5).astype('uint8').reshape(n, 28, 28)

    filas = int(np.ceil(np.sqrt(n)))
    fig, axes = plt.subplots(filas, filas, figsize=(filas, filas))
    for i, ax in enumerate(axes.flatten()):
        if i < n:
            ax.imshow(imagenes[i], cmap='gray')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    return imagenes

imagenes_sinteticas = mostrar_generadas(16)

## 5. Balancear el dataset de entrenamiento

Las imágenes sintéticas generadas se añaden al conjunto real de la clase minoritaria antes de entrenar (o reentrenar) el clasificador de defectos, reduciendo el desbalance sin esperar a acumular más ejemplos reales.

```python
x_clase_balanceada = np.concatenate([x_clase, imagenes_sinteticas_normalizadas], axis=0)
# usar x_clase_balanceada junto con el resto de clases para entrenar el clasificador final
```

## 6. Limitaciones

- Con pocas épocas de entrenamiento, las imágenes sintéticas pueden ser reconociblemente artificiales — para producción se recomienda entrenar más épocas y validar con un clasificador auxiliar que las imágenes sintéticas realmente ayudan (y no introducen ruido).
- Una GAN aprende la distribución de los ejemplos reales disponibles: si esos pocos ejemplos no cubren toda la variabilidad real de los defectos, las imágenes sintéticas heredarán ese sesgo.
- Alternativas a considerar según el caso: técnicas de aumento de datos clásicas (rotación, ruido, recorte) como primer paso más simple y barato antes de justificar una GAN completa.